In [8]:
import pandas as pd

data = r"C:\dev\Paddy_Prediction\data\raw\paddydataset.csv"
df = pd.read_csv(data)

In [9]:
#separating features and target variable
X = df.drop('Paddy yield(in Kg)', axis=1)
y = df['Paddy yield(in Kg)']

print(f"Dataset shape: {X.shape}")
print(f"Target shape: {y.shape}")

Dataset shape: (2789, 44)
Target shape: (2789,)


Defining feature groups

In [10]:
# Compute feature groups from X (exclude the target column)
categorical_features = X.select_dtypes(include=["object"]).columns
integer_features = X.select_dtypes(include=["int64"]).columns
float_features = X.select_dtypes(include=["float64"]).columns


C:\Users\soham_ai\AppData\Local\Temp\ipykernel_18140\3148805783.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns


In [11]:
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Integer features ({len(integer_features)}): {integer_features}")
print(f"Float features ({len(float_features)}): {float_features}")

Categorical features (8): Index(['Agriblock', 'Variety', 'Soil Types', 'Nursery',
       'Wind Direction_D1_D30', 'Wind Direction_D31_D60',
       'Wind Direction_D61_D90', 'Wind Direction_D91_D120'],
      dtype='str')
Integer features (18): Index(['Hectares ', 'Seedrate(in Kg)', 'Nursery area (Cents)',
       'LP_nurseryarea(in Tonnes)', 'DAP_20days', 'Weed28D_thiobencarb',
       'Micronutrients_70Days', 'Pest_60Day(in ml)', 'Max temp_D1_D30',
       'Max temp_D31_D60', 'Inst Wind Speed_D1_D30(in Knots)',
       'Inst Wind Speed_D31_D60(in Knots)',
       'Inst Wind Speed_D61_D90(in Knots)',
       'Inst Wind Speed_D91_D120(in Knots)', 'Relative Humidity_D31_D60',
       'Relative Humidity_D61_D90', 'Relative Humidity_D91_D120',
       'Trash(in bundles)'],
      dtype='str')
Float features (18): Index(['LP_Mainfield(in Tonnes)', 'Urea_40Days', 'Potassh_50Days',
       '30DRain( in mm)', '30DAI(in mm)', '30_50DRain( in mm)',
       '30_50DAI(in mm)', '51_70DRain(in mm)', '51_70AI(in

Creating a preprocessing pipeline

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Create preprocessing pipeline
preprocessor_tree = ColumnTransformer(
    transformers=[
        ('continuous', StandardScaler(), float_features),  # Scale continuous features
        ('integer', 'passthrough', integer_features),  # No scaling
        ('categorical', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ],
    remainder='drop'
)

X_tree = preprocessor_tree.fit_transform(X)


In [13]:
def get_feature_names_tree(preprocessor, continuous_features, integer_features, categorical_features):
    # Continuous features (standardized)
    cont_names = list(continuous_features)
    
    # Integer features (unchanged)
    int_names = list(integer_features)
    
    # Categorical features (one-hot encoded)
    cat_names = []
    ohe = preprocessor.named_transformers_['categorical']
    for i, cat_col in enumerate(categorical_features):
        categories = ohe.categories_[i][1:]  # Drop first
        cat_names.extend([f"{cat_col}_{cat}" for cat in categories])
    
    return cont_names + int_names + cat_names
 
feature_names_tree = get_feature_names_tree(
    preprocessor_tree, float_features, integer_features, categorical_features
)
 
# Create DataFrame
X_tree_df = pd.DataFrame(X_tree, columns=feature_names_tree, index=X.index)
df_tree = X_tree_df.copy()
df_tree['Paddy yield(in Kg)'] = y.values

In [14]:
#after one-hot encoding, the no of features increased from 45 to 64

In [15]:
df_tree.shape

(2789, 64)

In [16]:
df_tree.columns

Index(['LP_Mainfield(in Tonnes)', 'Urea_40Days', 'Potassh_50Days',
       '30DRain( in mm)', '30DAI(in mm)', '30_50DRain( in mm)',
       '30_50DAI(in mm)', '51_70DRain(in mm)', '51_70AI(in mm)',
       '71_105DRain(in mm)', '71_105DAI(in mm)', 'Min temp_D1_D30',
       'Min temp_D31_D60', 'Min temp_D61_D90', 'Max temp_D61_D90',
       'Min temp_D91_D120', 'Max temp_D91_D120', 'Relative Humidity_D1_D30',
       'Hectares ', 'Seedrate(in Kg)', 'Nursery area (Cents)',
       'LP_nurseryarea(in Tonnes)', 'DAP_20days', 'Weed28D_thiobencarb',
       'Micronutrients_70Days', 'Pest_60Day(in ml)', 'Max temp_D1_D30',
       'Max temp_D31_D60', 'Inst Wind Speed_D1_D30(in Knots)',
       'Inst Wind Speed_D31_D60(in Knots)',
       'Inst Wind Speed_D61_D90(in Knots)',
       'Inst Wind Speed_D91_D120(in Knots)', 'Relative Humidity_D31_D60',
       'Relative Humidity_D61_D90', 'Relative Humidity_D91_D120',
       'Trash(in bundles)', 'Agriblock_Cuddalore', 'Agriblock_Kallakurichi',
       'Agribloc

In [59]:
import os

df_tree.to_csv('../data/transformed/paddy_preprocessed_tree.csv', index=False)

Pre-processing for Classic Models

In [ ]:
preprocessor_all = ColumnTransformer(
    transformers=[
        ('continuous', StandardScaler(), float_features),
        ('integer', StandardScaler(), integer_features),  # scaled this time
        ('categorical', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ],
    remainder='drop'
)

X_lasso = preprocessor_all.fit_transform(X)

# Reuse same naming function
feature_names_lasso = get_feature_names_tree(
    preprocessor_all, float_features, integer_features, categorical_features
)

X_lasso_df = pd.DataFrame(X_lasso, columns=feature_names_lasso, index=X.index)
df_lasso = X_lasso_df.copy()
df_lasso['Paddy yield(in Kg)'] = y.values

print(df_lasso.shape)
df_lasso.to_csv('../data/transformed/paddy_preprocessed.csv', index=False)